In [1]:
import numpy as np
import pandas as pd


def make_mock_risk_data(
    n_samples=100_000,
    random_state=2026,
):
    """
    构造用于风控分析测试的模拟数据。

    数据结构：
    - user_id: 唯一用户ID
    - dt: 日期，格式 YYYY-MM-DD
    - dataset: train / test / oot
    - user_type: level_1 / level_2 / level_3
    - 1m_30: 0/1风险标签，1=坏人，0=好人
    - new_score: 新模型分，越高风险越高
    - old_score: 老模型分，越高风险越高

    数据集比例：
    - train: 54%
    - test: 36%
    - oot: 10%
    """

    rng = np.random.default_rng(random_state)

    # ============================================================
    # 1. 样本数量
    # ============================================================
    n_oot = int(n_samples * 0.10)
    n_dev = n_samples - n_oot

    # train:test = 6:4
    n_train = int(n_dev * 0.60)
    n_test = n_dev - n_train

    print(f"总样本数: {n_samples:,}")
    print(f"train:   {n_train:,}")
    print(f"test:    {n_test:,}")
    print(f"oot:     {n_oot:,}")

    # ============================================================
    # 2. user_id
    # ============================================================
    user_id = [
        f"user_{i:06d}"
        for i in range(1, n_samples + 1)
    ]

    # ============================================================
    # 3. dataset
    # ============================================================
    #
    # 先构造 90% 开发样本：
    # train:test = 6:4
    #
    dev_dataset = np.array(
        ["train"] * n_train
        + ["test"] * n_test
    )

    # 随机打乱 train / test
    rng.shuffle(dev_dataset)

    # 后 10% 作为 OOT
    dataset = np.concatenate([
        dev_dataset,
        np.repeat("oot", n_oot)
    ])

    # ============================================================
    # 4. dt
    # ============================================================
    #
    # train/test:
    # 2025-01-01 ~ 2026-06-30
    #
    # oot:
    # 2026-07-01 ~ 2026-09-01
    #

    dev_dates = pd.date_range(
        "2025-01-01",
        "2026-06-30",
        freq="D"
    )

    oot_dates = pd.date_range(
        "2026-07-01",
        "2026-09-01",
        freq="D"
    )

    dt_dev = rng.choice(
        dev_dates,
        size=n_dev,
        replace=True
    )

    dt_oot = rng.choice(
        oot_dates,
        size=n_oot,
        replace=True
    )

    dt = np.concatenate([
        dt_dev,
        dt_oot
    ])

    # ============================================================
    # 5. user_type
    # ============================================================
    #
    # 假设用户等级占比：
    #
    # level_1 = 45%
    # level_2 = 35%
    # level_3 = 20%
    #

    user_type = rng.choice(
        ["level_1", "level_2", "level_3"],
        size=n_samples,
        p=[0.45, 0.35, 0.20]
    )

    # ============================================================
    # 6. 生成真实风险概率
    # ============================================================
    #
    # 不同 user_type 风险水平不同：
    #
    # level_1 风险最低
    # level_2 中等
    # level_3 风险最高
    #

    type_effect = pd.Series(user_type).map({
        "level_1": -0.8,
        "level_2": 0.0,
        "level_3": 0.8
    }).to_numpy()

    # 每个人还有一个隐藏风险因子
    latent_risk = rng.normal(
        loc=0,
        scale=1,
        size=n_samples
    )

    # OOT 稍微发生一点风险漂移
    oot_effect = np.where(
        dataset == "oot",
        0.20,
        0
    )

    # logit
    logit = (
        -2.2
        + 0.80 * latent_risk
        + 0.50 * type_effect
        + oot_effect
    )

    bad_probability = 1 / (
        1 + np.exp(-logit)
    )

    # ============================================================
    # 7. 生成标签
    # ============================================================

    target = rng.binomial(
        n=1,
        p=bad_probability
    )

    # ============================================================
    # 8. 生成模型分数
    # ============================================================
    #
    # 两个模型都捕捉 latent_risk，
    # 但 new_score 噪声更小，
    # 因此理论上 new_score AUC 会略高。
    #

    new_model_signal = (
        latent_risk
        + 0.45 * type_effect
        + rng.normal(0, 0.65, n_samples)
    )

    old_model_signal = (
        latent_risk
        + 0.35 * type_effect
        + rng.normal(0, 0.95, n_samples)
    )

    # 转成比较像实际模型的 300 ~ 900 分
    def scale_score(x):
        percentile = pd.Series(x).rank(
            pct=True
        ).to_numpy()

        return np.round(
            300 + percentile * 600,
            2
        )

    new_score = scale_score(new_model_signal)
    old_score = scale_score(old_model_signal)

    # ============================================================
    # 9. 生成 DataFrame
    # ============================================================

    df = pd.DataFrame({
        "user_id": user_id,
        "dt": pd.to_datetime(dt).strftime("%Y-%m-%d"),
        "dataset": dataset,
        "user_type": user_type,
        "1m_30": target,
        "new_score": new_score,
        "old_score": old_score,
    })

    return df

In [2]:
df = make_mock_risk_data(
    n_samples=100_000,
    random_state=2026,
)

总样本数: 100,000
train:   54,000
test:    36,000
oot:     10,000


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 7 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   user_id    100000 non-null  str    
 1   dt         100000 non-null  str    
 2   dataset    100000 non-null  str    
 3   user_type  100000 non-null  str    
 4   1m_30      100000 non-null  int64  
 5   new_score  100000 non-null  float64
 6   old_score  100000 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 5.3 MB


In [4]:
df.head()

,user_id,dt,dataset,user_type,1m_30,new_score,old_score
0,user_000001,2026-06-26,train,level_1,0,378.08,527.27
1,user_000002,2025-06-29,train,level_2,0,676.47,508.43
2,user_000003,2025-08-19,train,level_2,0,427.19,594.23
3,user_000004,2026-02-15,train,level_1,0,592.66,812.54
4,user_000005,2026-03-22,train,level_2,0,496.56,387.83


In [5]:
df.columns.tolist()

['user_id', 'dt', 'dataset', 'user_type', '1m_30', 'new_score', 'old_score']

In [6]:
from phl_risk.analysis import (
    AUC,
    KS,
    BinDimension,
    Count,
    Cube,
    EventRate,
    QuantileBinner,
    Share,
)

In [7]:
cube = Cube(["dt", "dataset"], [AUC("new_score", "1m_30"), KS("new_score", "1m_30")])

In [8]:
result = cube.compute(df,totals=True)

In [10]:
result.layout(
    rows=["dt"],
    columns=["metric", "dataset"],
    totals=["dt"],
    total_label="总计",
)

metric     auc__new_score                     ks__new_score            \
dataset             train      test       oot         train      test   
dt                                                                      
2026-06-26       0.779070  0.704261       NaN      0.552326  0.413534   
2025-06-29       0.635464  0.678005       NaN      0.273627  0.471655   
2025-08-19       0.683794  0.683516       NaN      0.361660  0.417582   
2026-02-15       0.588235  0.754098       NaN      0.317647  0.487705   
2026-03-22       0.789011  0.717014       NaN      0.445055  0.437500   
...                   ...       ...       ...           ...       ...   
2026-07-16            NaN       NaN  0.776539           NaN       NaN   
2026-08-28            NaN       NaN  0.716379           NaN       NaN   
2026-08-31            NaN       NaN  0.793750           NaN       NaN   
2026-08-20            NaN       NaN  0.637097           NaN       NaN   
总计               0.685440  0.695457  0.681893      0.270359  0.292304   

metric                
dataset          oot  
dt                    
2026-06-26       NaN  
2025-06-29       NaN  
2025-08-19       NaN  
2026-02-15       NaN  
2026-03-22       NaN  
...              ...  
2026-07-16  0.518287  
2026-08-28  0.392465  
2026-08-31  0.541667  
2026-08-20  0.253857  
总计          0.270516  

[610 rows x 6 columns]

In [13]:
# result.layout().unstack(level='dataset')

result.layout(
    rows=["dt"],
    columns=["metric", "dataset"],
    totals=["dataset","dt"],
    total_label="总计",
)

metric     auc__new_score                               ks__new_score  \
dataset             train      test       oot        总计         train   
dt                                                                      
2026-06-26       0.779070  0.704261       NaN  0.741627      0.552326   
2025-06-29       0.635464  0.678005       NaN  0.668000      0.273627   
2025-08-19       0.683794  0.683516       NaN  0.677282      0.361660   
2026-02-15       0.588235  0.754098       NaN  0.656012      0.317647   
2026-03-22       0.789011  0.717014       NaN  0.736429      0.445055   
...                   ...       ...       ...       ...           ...   
2026-07-16            NaN       NaN  0.776539  0.776539           NaN   
2026-08-28            NaN       NaN  0.716379  0.716379           NaN   
2026-08-31            NaN       NaN  0.793750  0.793750           NaN   
2026-08-20            NaN       NaN  0.637097  0.637097           NaN   
总计               0.685440  0.695457  0.681893  0.688626      0.270359   

metric                                    
dataset         test       oot        总计  
dt                                        
2026-06-26  0.413534       NaN  0.449761  
2025-06-29  0.471655       NaN  0.333333  
2025-08-19  0.417582       NaN  0.348195  
2026-02-15  0.487705       NaN  0.295282  
2026-03-22  0.437500       NaN  0.386527  
...              ...       ...       ...  
2026-07-16       NaN  0.518287  0.518287  
2026-08-28       NaN  0.392465  0.392465  
2026-08-31       NaN  0.541667  0.541667  
2026-08-20       NaN  0.253857  0.253857  
总计          0.292304  0.270516  0.277170  

[610 rows x 8 columns]

In [14]:
print(cube.explain())

      Section                    Item                         Description
0   Execution                  Engine                              pandas
1   Execution                Grouping                        dt × dataset
2   Dimension                      dt                          Source: dt
3   Dimension                 dataset                     Source: dataset
4     Measure          auc__new_score  AUC: target=1m_30, score=new_score
5     Measure           ks__new_score   KS: target=1m_30, score=new_score
6   Execution       Shared aggregates                                   0
7   Execution           Group metrics                                   2
8   Execution                 Filters                                   0
9   Execution                  Totals                            Disabled
10     Policy       Dimension missing                                drop
11     Policy  Target / score missing             Drop within each metric
12     Policy          Invalid metric 

In [36]:
df.columns.tolist()

['user_id', 'dt', 'dataset', 'user_type', '1m_30', 'new_score', 'old_score']

In [27]:
df.user_type.value_counts().index.tolist()

['level_1', 'level_2', 'level_3']

In [ ]:
result = cube.compute(oot, totals=True)



In [15]:
oot = df[df["dataset"] == "oot"]
cube = Cube(
    [BinDimension("new_score", QuantileBinner(5)), BinDimension("old_score", QuantileBinner(5))],
    [Count(), Share(), EventRate("1m_30", name="bad_rate")],
).fit(oot)

In [20]:
learned = [dimension.transformer.bin_edges_ for dimension in cube.dimensions_]
result = cube.compute(oot, totals=True)

vertical = result.layout(
    rows=["metric", "new_score_bin"],
    columns=["old_score_bin"],
    totals=True,
    total_label="总计",
    bin_labels="interval",
)

vertical

old_score_bin                [-inf, 416.916]  (416.916, 541.586]  \
metric   new_score_bin                                             
count    [-inf, 422.516]         1015.000000          533.000000   
         (422.516, 537.296]       518.000000          607.000000   
         (537.296, 658.024]       303.000000          441.000000   
         (658.024, 781.11]        122.000000          292.000000   
         (781.11, inf]             42.000000          127.000000   
         总计                      2000.000000         2000.000000   
share    [-inf, 422.516]            0.101500            0.053300   
         (422.516, 537.296]         0.051800            0.060700   
         (537.296, 658.024]         0.030300            0.044100   
         (658.024, 781.11]          0.012200            0.029200   
         (781.11, inf]              0.004200            0.012700   
         总计                         0.200000            0.200000   
bad_rate [-inf, 422.516]            0.032512            0.046904   
         (422.516, 537.296]         0.052124            0.077430   
         (537.296, 658.024]         0.105611            0.099773   
         (658.024, 781.11]          0.106557            0.109589   
         (781.11, inf]              0.119048            0.188976   
         总计                         0.055000            0.086000   

old_score_bin                (541.586, 661.33]  (661.33, 778.4760000000001]  \
metric   new_score_bin                                                        
count    [-inf, 422.516]            277.000000                   128.000000   
         (422.516, 537.296]         447.000000                   313.000000   
         (537.296, 658.024]         525.000000                   458.000000   
         (658.024, 781.11]          452.000000                   581.000000   
         (781.11, inf]              299.000000                   520.000000   
         总计                        2000.000000                  2000.000000   
share    [-inf, 422.516]              0.027700                     0.012800   
         (422.516, 537.296]           0.044700                     0.031300   
         (537.296, 658.024]           0.052500                     0.045800   
         (658.024, 781.11]            0.045200                     0.058100   
         (781.11, inf]                0.029900                     0.052000   
         总计                           0.200000                     0.200000   
bad_rate [-inf, 422.516]              0.090253                     0.078125   
         (422.516, 537.296]           0.116331                     0.067093   
         (537.296, 658.024]           0.112381                     0.104803   
         (658.024, 781.11]            0.146018                     0.173838   
         (781.11, inf]                0.183946                     0.234615   
         总计                           0.128500                     0.151000   

old_score_bin                (778.4760000000001, inf]          总计  
metric   new_score_bin                                             
count    [-inf, 422.516]                    47.000000   2000.0000  
         (422.516, 537.296]                115.000000   2000.0000  
         (537.296, 658.024]                273.000000   2000.0000  
         (658.024, 781.11]                 553.000000   2000.0000  
         (781.11, inf]                    1012.000000   2000.0000  
         总计                               2000.000000  10000.0000  
share    [-inf, 422.516]                     0.004700      0.2000  
         (422.516, 537.296]                  0.011500      0.2000  
         (537.296, 658.024]                  0.027300      0.2000  
         (658.024, 781.11]                   0.055300      0.2000  
         (781.11, inf]                       0.101200      0.2000  
         总计                                  0.200000      1.0000  
bad_rate [-inf, 422.516]                     0.127660      0.0495  
         (422.516, 537.296]           

In [ ]:
horizontal = result.layout(
    rows=["old_score_bin"],
    columns=["metric", "new_score_bin"],
    # totals=True,
    # total_label="总计",
    # bin_labels="interval",
)

horizontal

metric          count                                share                  \
new_score_bin      B1     B2     B3     B4      B5      B1      B2      B3   
old_score_bin                                                                
B1             1015.0  518.0  303.0  122.0    42.0  0.1015  0.0518  0.0303   
B2              533.0  607.0  441.0  292.0   127.0  0.0533  0.0607  0.0441   
B3              277.0  447.0  525.0  452.0   299.0  0.0277  0.0447  0.0525   
B4              128.0  313.0  458.0  581.0   520.0  0.0128  0.0313  0.0458   
B5               47.0  115.0  273.0  553.0  1012.0  0.0047  0.0115  0.0273   

metric                         bad_rate                                \
new_score_bin      B4      B5        B1        B2        B3        B4   
old_score_bin                                                           
B1             0.0122  0.0042  0.032512  0.052124  0.105611  0.106557   
B2             0.0292  0.0127  0.046904  0.077430  0.099773  0.109589   
B3             0.0452  0.0299  0.090253  0.116331  0.112381  0.146018   
B4             0.0581  0.0520  0.078125  0.067093  0.104803  0.173838   
B5             0.0553  0.1012  0.127660  0.121739  0.186813  0.202532   

metric                   
new_score_bin        B5  
old_score_bin            
B1             0.119048  
B2             0.188976  
B3             0.183946  
B4             0.234615  
B5             0.305336

In [38]:
cube = Cube(["dt", "user_type"], [AUC("new_score", "1m_30"), KS("new_score", "1m_30")])


result_train = cube.compute(df[df['dataset']=='train'],totals=True)
result_test = cube.compute(df[df['dataset']=='test'],totals=True)
result_oot = cube.compute(df[df['dataset']=='oot'],totals=True)


layout_train = result_train.layout(
    rows=["dt"],
    columns=["metric", "user_type"],
    totals=["dt"],
    total_label="总计",
)
layout_test = result_test.layout(
    rows=["dt"],
    columns=["metric", "user_type"],
    totals=["dt"],
    total_label="总计",
)
layout_oot = result_oot.layout(
    rows=["dt"],
    columns=["metric", "user_type"],
    totals=["dt"],
    total_label="总计",
)

In [46]:
layout_oot

metric     auc__new_score                     ks__new_score            \
user_type         level_1   level_2   level_3       level_1   level_2   
dt                                                                      
2026-07-08       0.759328  0.531136  0.752688      0.541045  0.197802   
2026-08-14       0.611529  0.604651  0.689655      0.325815  0.305233   
2026-07-06       0.521154  0.682353  0.617647      0.169231  0.360784   
2026-08-05       0.577259  0.290123  0.500000      0.387755  0.407407   
2026-08-04       0.745718  0.740000  0.814815      0.513834  0.677778   
...                   ...       ...       ...           ...       ...   
2026-07-16       0.873333  0.666667  0.640625      0.766667  0.357143   
2026-08-28       0.644118  0.770000  0.741379      0.417647  0.640000   
2026-08-31       0.854864  0.779605  0.728000      0.681021  0.592105   
2026-08-20       0.655232  0.613475  0.618056      0.349914  0.436170   
总计               0.687284  0.656505  0.672128      0.281114  0.238988   

metric                
user_type    level_3  
dt                    
2026-07-08  0.478495  
2026-08-14  0.522989  
2026-07-06  0.352941  
2026-08-05  0.353659  
2026-08-04  0.666667  
...              ...  
2026-07-16  0.375000  
2026-08-28  0.626437  
2026-08-31  0.440000  
2026-08-20  0.416667  
总计          0.256899  

[64 rows x 6 columns]

In [44]:
pd.concat([layout_train, layout_test, layout_oot], axis=1, ignore_index=False)

metric     auc__new_score                     ks__new_score            \
user_type         level_1   level_2   level_3       level_1   level_2   
dt                                                                      
2026-06-26       0.928571  0.741379  0.683333      0.928571  0.448276   
2025-06-29       0.503876  0.953488  0.466667      0.286822  0.860465   
2025-08-19       0.478261  0.725962  0.600000      0.326087  0.509615   
2026-02-15       0.585938  0.897436       NaN      0.375000  0.897436   
2026-03-22       0.750000  0.625000  0.916667      0.474359  0.500000   
...                   ...       ...       ...           ...       ...   
2026-08-10            NaN       NaN       NaN           NaN       NaN   
2026-07-16            NaN       NaN       NaN           NaN       NaN   
2026-08-28            NaN       NaN       NaN           NaN       NaN   
2026-08-31            NaN       NaN       NaN           NaN       NaN   
2026-08-20            NaN       NaN       NaN           NaN       NaN   

metric               auc__new_score                     ks__new_score  \
user_type    level_3        level_3   level_1   level_2       level_3   
dt                                                                      
2026-06-26  0.550000       0.452381  0.728395  1.000000      0.452381   
2025-06-29  0.333333       0.416667  0.800000  0.611765      0.583333   
2025-08-19  0.600000       0.416667  0.705128  0.740741      0.583333   
2026-02-15       NaN       0.777778  0.718750       NaN      0.555556   
2026-03-22  0.800000       0.809524  0.525641  0.755556      0.714286   
...              ...            ...       ...       ...           ...   
2026-08-10       NaN            NaN       NaN       NaN           NaN   
2026-07-16       NaN            NaN       NaN       NaN           NaN   
2026-08-28       NaN            NaN       NaN       NaN           NaN   
2026-08-31       NaN            NaN       NaN       NaN           NaN   
2026-08-20       NaN            NaN       NaN       NaN           NaN   

metric                         auc__new_score                      \
user_type    level_1   level_2        level_1   level_2   level_3   
dt                                                                  
2026-06-26  0.592593  1.000000            NaN       NaN       NaN   
2025-06-29  0.800000  0.447059            NaN       NaN       NaN   
2025-08-19  0.551282  0.555556            NaN       NaN       NaN   
2026-02-15  0.531250       NaN            NaN       NaN       NaN   
2026-03-22  0.282051  0.500000            NaN       NaN       NaN   
...              ...       ...            ...       ...       ...   
2026-08-10       NaN       NaN       0.663580  0.513889  0.564286   
2026-07-16       NaN       NaN       0.873333  0.666667  0.640625   
2026-08-28       NaN       NaN       0.644118  0.770000  0.741379   
2026-08-31       NaN       NaN       0.854864  0.779605  0.728000   
2026-08-20       NaN       NaN       0.655232  0.613475  0.618056   

metric     ks__new_score                      
user_type        level_1   level_2   level_3  
dt                                            
2026-06-26           NaN       NaN       NaN  
2025-06-29           NaN       NaN       NaN  
2025-08-19           NaN       NaN       NaN  
2026-02-15           NaN       NaN       NaN  
2026-03-22           NaN       NaN       NaN  
...                  ...       ...       ...  
2026-08-10      0.407407  0.229167  0.292857  
2026-07-16      0.766667  0.357143  0.375000  
2026-08-28      0.417647  0.640000  0.626437  
2026-08-31      0.681021  0.592105  0.440000  
2026-08-20      0.349914  0.436170  0.416667  

[610 rows x 18 columns]

In [50]:
layout_train.loc['总计']

metric          user_type
auc__new_score  level_1      0.677158
                level_2      0.670007
                level_3      0.673943
ks__new_score   level_1      0.257988
                level_2      0.254753
                level_3      0.247270
Name: 总计, dtype: float64

In [51]:
layout_test.loc['总计']

metric          user_type
auc__new_score  level_3      0.682857
                level_1      0.689303
                level_2      0.681204
ks__new_score   level_3      0.284400
                level_1      0.281675
                level_2      0.267566
Name: 总计, dtype: float64

In [52]:
layout_oot.loc['总计']

metric          user_type
auc__new_score  level_1      0.687284
                level_2      0.656505
                level_3      0.672128
ks__new_score   level_1      0.281114
                level_2      0.238988
                level_3      0.256899
Name: 总计, dtype: float64

In [57]:
concat_result = pd.concat(
    [layout_train, layout_test, layout_oot], 
    axis=1, 
    keys=['Train', 'Test', 'OOT'],  # 给这三块分别起个名字
    ignore_index=False              # 保留原有的列结构
)

In [58]:
concat_result

Train                                              \
metric     auc__new_score                     ks__new_score             
user_type         level_1   level_2   level_3       level_1   level_2   
dt                                                                      
2026-06-26       0.928571  0.741379  0.683333      0.928571  0.448276   
2025-06-29       0.503876  0.953488  0.466667      0.286822  0.860465   
2025-08-19       0.478261  0.725962  0.600000      0.326087  0.509615   
2026-02-15       0.585938  0.897436       NaN      0.375000  0.897436   
2026-03-22       0.750000  0.625000  0.916667      0.474359  0.500000   
...                   ...       ...       ...           ...       ...   
2026-08-10            NaN       NaN       NaN           NaN       NaN   
2026-07-16            NaN       NaN       NaN           NaN       NaN   
2026-08-28            NaN       NaN       NaN           NaN       NaN   
2026-08-31            NaN       NaN       NaN           NaN       NaN   
2026-08-20            NaN       NaN       NaN           NaN       NaN   

                               Test                                    \
metric               auc__new_score                     ks__new_score   
user_type    level_3        level_3   level_1   level_2       level_3   
dt                                                                      
2026-06-26  0.550000       0.452381  0.728395  1.000000      0.452381   
2025-06-29  0.333333       0.416667  0.800000  0.611765      0.583333   
2025-08-19  0.600000       0.416667  0.705128  0.740741      0.583333   
2026-02-15       NaN       0.777778  0.718750       NaN      0.555556   
2026-03-22  0.800000       0.809524  0.525641  0.755556      0.714286   
...              ...            ...       ...       ...           ...   
2026-08-10       NaN            NaN       NaN       NaN           NaN   
2026-07-16       NaN            NaN       NaN       NaN           NaN   
2026-08-28       NaN            NaN       NaN       NaN           NaN   
2026-08-31       NaN            NaN       NaN       NaN           NaN   
2026-08-20       NaN            NaN       NaN       NaN           NaN   

                                          OOT                      \
metric                         auc__new_score                       
user_type    level_1   level_2        level_1   level_2   level_3   
dt                                                                  
2026-06-26  0.592593  1.000000            NaN       NaN       NaN   
2025-06-29  0.800000  0.447059            NaN       NaN       NaN   
2025-08-19  0.551282  0.555556            NaN       NaN       NaN   
2026-02-15  0.531250       NaN            NaN       NaN       NaN   
2026-03-22  0.282051  0.500000            NaN       NaN       NaN   
...              ...       ...            ...       ...       ...   
2026-08-10       NaN       NaN       0.663580  0.513889  0.564286   
2026-07-16       NaN       NaN       0.873333  0.666667  0.640625   
2026-08-28       NaN       NaN       0.644118  0.770000  0.741379   
2026-08-31       NaN       NaN       0.854864  0.779605  0.728000   
2026-08-20       NaN       NaN       0.655232  0.613475  0.618056   

                                              
metric     ks__new_score                      
user_type        level_1   level_2   level_3  
dt                                            
2026-06-26           NaN       NaN       NaN  
2025-06-29           NaN       NaN       NaN  
2025-08-19           NaN       NaN       NaN  
2026-02-15           NaN       NaN       NaN  
2026-03-22           NaN       NaN       NaN  
...                  ...       ...       ...  
2026-08-10      0.407407  0.229167  0.292857  
2026-07-16      0.766667  0.357143  0.375000  
2026-08-28      0.417647  0.640000  0.626437  
2026-08-31      0.681021  0.592105  0.440000  
2026-08-20      0.349914  0.436170  0.416667  

[610 rows x 18 columns]

In [59]:
concat_result = pd.concat(
    [layout_train, layout_test, layout_oot], 
    axis=1, 
    keys=['Train', 'Test', 'OOT'],  # 给这三块分别起个名字
    ignore_index=False              # 保留原有的列结构
)
total_row = concat_result.loc[['总计']]
concat_result = pd.concat([concat_result.drop('总计'), total_row])

In [60]:
concat_result

Train                                              \
metric     auc__new_score                     ks__new_score             
user_type         level_1   level_2   level_3       level_1   level_2   
dt                                                                      
2026-06-26       0.928571  0.741379  0.683333      0.928571  0.448276   
2025-06-29       0.503876  0.953488  0.466667      0.286822  0.860465   
2025-08-19       0.478261  0.725962  0.600000      0.326087  0.509615   
2026-02-15       0.585938  0.897436       NaN      0.375000  0.897436   
2026-03-22       0.750000  0.625000  0.916667      0.474359  0.500000   
...                   ...       ...       ...           ...       ...   
2026-07-16            NaN       NaN       NaN           NaN       NaN   
2026-08-28            NaN       NaN       NaN           NaN       NaN   
2026-08-31            NaN       NaN       NaN           NaN       NaN   
2026-08-20            NaN       NaN       NaN           NaN       NaN   
总计               0.677158  0.670007  0.673943      0.257988  0.254753   

                               Test                                    \
metric               auc__new_score                     ks__new_score   
user_type    level_3        level_3   level_1   level_2       level_3   
dt                                                                      
2026-06-26  0.550000       0.452381  0.728395  1.000000      0.452381   
2025-06-29  0.333333       0.416667  0.800000  0.611765      0.583333   
2025-08-19  0.600000       0.416667  0.705128  0.740741      0.583333   
2026-02-15       NaN       0.777778  0.718750       NaN      0.555556   
2026-03-22  0.800000       0.809524  0.525641  0.755556      0.714286   
...              ...            ...       ...       ...           ...   
2026-07-16       NaN            NaN       NaN       NaN           NaN   
2026-08-28       NaN            NaN       NaN       NaN           NaN   
2026-08-31       NaN            NaN       NaN       NaN           NaN   
2026-08-20       NaN            NaN       NaN       NaN           NaN   
总计          0.247270       0.682857  0.689303  0.681204      0.284400   

                                          OOT                      \
metric                         auc__new_score                       
user_type    level_1   level_2        level_1   level_2   level_3   
dt                                                                  
2026-06-26  0.592593  1.000000            NaN       NaN       NaN   
2025-06-29  0.800000  0.447059            NaN       NaN       NaN   
2025-08-19  0.551282  0.555556            NaN       NaN       NaN   
2026-02-15  0.531250       NaN            NaN       NaN       NaN   
2026-03-22  0.282051  0.500000            NaN       NaN       NaN   
...              ...       ...            ...       ...       ...   
2026-07-16       NaN       NaN       0.873333  0.666667  0.640625   
2026-08-28       NaN       NaN       0.644118  0.770000  0.741379   
2026-08-31       NaN       NaN       0.854864  0.779605  0.728000   
2026-08-20       NaN       NaN       0.655232  0.613475  0.618056   
总计          0.281675  0.267566       0.687284  0.656505  0.672128   

                                              
metric     ks__new_score                      
user_type        level_1   level_2   level_3  
dt                                            
2026-06-26           NaN       NaN       NaN  
2025-06-29           NaN       NaN       NaN  
2025-08-19           NaN       NaN       NaN  
2026-02-15           NaN       NaN       NaN  
2026-03-22           NaN       NaN       NaN  
...                  ...       ...       ...  
2026-07-16      0.766667  0.357143  0.375000  
2026-08-28      0.417647  0.640000  0.626437  
2026-08-31      0.681021  0.592105  0.440000  
2026-08-20      0.349914  0.436170  0.416667  
总计              0.281114  0.238988  0.256899  

[610 rows x 18 columns]